<a href="https://colab.research.google.com/github/VILLAROSA-eng/FUNDAI-Laboratories-VILLAROSA/blob/main/Lab4_Logic_KR_Villarosa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Logic and Knowledge Representation

## Fundamentals of Artificial Intelligence

**Name:** Jojant G. Villarosa
**Course:** BSCS-AI
**Section:** 09282-FUNDAI
**Date:** September 15, 2026

**GitHub URL:** https://github.com/VILLAROSA-eng/FUNDAI-Laboratories-VILLAROSA

## Description
This laboratory uses Python and SymPy to perform truth table generation, satisfiability checking, theorem proving, and logical deduction.

In [ ]:
from sympy import symbols, And, Or, Not, Implies, Equivalent, satisfiable
from itertools import product

In [ ]:
P, Q, R = symbols(' P Q R')

In [ ]:
def print_truth_table(expression, symbol_list):
    header = [str(s) for s in symbol_list] + [str(expression)]
    print(" | ".join(header))
    print("-" * (5 * len(header)))

    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        result = bool(expression.subs(mapping))
        row = [str(v) for v in values] + [str(result)]
        print(" | ".join(row))

    print()

In [ ]:
print("Negation: NOT P")
print_truth_table(Not(P), [P])

print("Conjunction: P AND Q")
print_truth_table(And(P, Q), [P, Q])

print("Disjunction: P OR Q")
print_truth_table(Or(P, Q), [P, Q])

print("Implication: P -> Q")
print_truth_table(Implies(P, Q), [P, Q])

print("Biconditional: P <-> Q")
print_truth_table(Equivalent(P, Q), [P, Q])

Negation: NOT P
P | ~P
----------
False | True
True | False

Conjunction: P AND Q
P | Q | P & Q
---------------
False | False | False
False | True | False
True | False | False
True | True | True

Disjunction: P OR Q
P | Q | P | Q
---------------
False | False | False
False | True | True
True | False | True
True | True | True

Implication: P -> Q
P | Q | Implies(P, Q)
---------------
False | False | True
False | True | True
True | False | False
True | True | True

Biconditional: P <-> Q
P | Q | Equivalent(P, Q)
---------------
False | False | True
False | True | False
True | False | False
True | True | True



In [ ]:
def is_tautology(expression, symbol_list):
    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        if not bool(expression.subs(mapping)):
            return False
    return True

law_of_excluded_middle = Or(P, Not(P))
contradiction = And(P, Not(P))
simple_implication = Implies(P, Q)

print("P OR NOT P is a tautology:", is_tautology(law_of_excluded_middle, [P]))
print("P AND NOT P is a tautology:", is_tautology(contradiction, [P]))
print("P -> Q is a tautology:", is_tautology(simple_implication, [P, Q]))

P OR NOT P is a tautology: True
P AND NOT P is a tautology: False
P -> Q is a tautology: False


In [ ]:
def is_satisfiable(expression):
    return satisfiable(expression) is not False

print("P AND NOT P is satisfiable:", is_satisfiable(And(P, Not(P))))
print("P OR Q is satisfiable:", is_satisfiable(Or(P, Q)))
print("P -> Q is satisfiable:", is_satisfiable(Implies(P, Q)))

P AND NOT P is satisfiable: False
P OR Q is satisfiable: True
P -> Q is satisfiable: True


In [ ]:
def to_conjunction(kb):
    if isinstance(kb, list):
        return And(*kb)
    return kb

def kb_entails(kb, conclusion):
    kb_expression = to_conjunction(kb)
    counter_check = And(kb_expression, Not(conclusion))
    return satisfiable(counter_check) is False

def check_entailment(kb, conclusion, label="Query"):
    holds = kb_entails(kb, conclusion)
    kb_expression = to_conjunction(kb)
    counterexample = satisfiable(And(kb_expression, Not(conclusion)))

    print(label)

    if holds:
        print("Result: Entailment holds.")
    else:
        print("Result: Entailment does not hold.")
        print("Counterexample model:", counterexample)

    print("-" * 60)
    return holds

Rain, Wet = symbols('Rain Wet')

kb_rain = [
    Implies(Rain, Wet),
    Rain
]

check_entailment(kb_rain, Wet, "Theorem Proving: Rain example")


Theorem Proving: Rain example
Result: Entailment holds.
------------------------------------------------------------


True

In [ ]:
kb_invalid = [
    Implies(Rain, Wet),
    Wet
]

check_entailment(kb_invalid, Rain, "Invalid Inference: Affirming the consequent")

Invalid Inference: Affirming the consequent
Result: Entailment does not hold.
Counterexample model: {Wet: True, Rain: False}
------------------------------------------------------------


False

In [ ]:
print("Logical Deduction Rules")
print("=" * 60)

# Modus Ponens
check_entailment(
    [P, Implies(P, Q)],
    Q,
    "Modus Ponens: P, P -> Q, therefore Q"
)

# Modus Tollens
check_entailment(
    [Not(Q), Implies(P, Q)],
    Not(P),
    "Modus Tollens: NOT Q, P -> Q, therefore NOT P"
)

Logical Deduction Rules
Modus Ponens: P, P -> Q, therefore Q
Result: Entailment holds.
------------------------------------------------------------
Modus Tollens: NOT Q, P -> Q, therefore NOT P
Result: Entailment holds.
------------------------------------------------------------


True

## Grounded First-Order Logic Example

Full First-Order Logic includes objects and quantifiers.
For this laboratory, we demonstrate a simple grounded FOL example
by converting FOL atoms into propositional symbols.

English:
- All humans are mortal.
- Socrates is human.
- Therefore, Socrates is mortal.

Grounded propositional form:
- Human_Socrates -> Mortal_Socrates

In [ ]:
Human_Socrates, Mortal_Socrates = symbols('Human_Socrates Mortal_Socrates')

kb_socrates = [
    Implies(Human_Socrates, Mortal_Socrates),
    Human_Socrates
]

check_entailment(
    kb_socrates,
    Mortal_Socrates,
    "Grounded FOL: Socrates is mortal"
)

Grounded FOL: Socrates is mortal
Result: Entailment holds.
------------------------------------------------------------


True

In [ ]:
def make_human_mortal_kb(constants):
    kb = []
    human = {}
    mortal = {}

    for name in constants:
        h, m = symbols(f'Human_{name} Mortal_{name}')
        human[name] = h
        mortal[name] = m
        kb.append(Implies(h, m))

    return kb, human, mortal

constants = ["Socrates", "Plato"]

kb_people, human, mortal = make_human_mortal_kb(constants)

# Add facts
kb_people.append(human["Socrates"])
kb_people.append(human["Plato"])

# Query: Is Plato mortal?
check_entailment(
    kb_people,
    mortal["Plato"],
    "Grounded FOL with multiple constants: Is Plato mortal?"
)

Grounded FOL with multiple constants: Is Plato mortal?
Result: Entailment holds.
------------------------------------------------------------


True

## Guide Questions and Answers

### 1. What is the difference between syntax and semantics?

**Answer:** Syntax refers to the rules and visual structure used to construct valid statements or expressions in a formal language (the "grammar"). Semantics defines the actual meaning, truth values, or interpretations assigned to those syntactically valid symbols and expressions in a given model.

### 2. Why is `P -> Q` true when `P` is false?

**Answer:** In classical logic, material implication $P \implies Q$ is defined as equivalent to $\neg P \lor Q$. When $P$ is false, the condition is satisfied vacuously because the statement makes a promise strictly contingent on $P$ being true; if $P$ never occurs, the overall conditional statement cannot be falsified.

### 3. What does it mean for a knowledge base to entail a conclusion?

**Answer:** A knowledge base ($KB$) entails a sentence/conclusion $\alpha$ (written as $KB \models \alpha$) if and only if $\alpha$ is true in every possible world or model where all statements in $KB$ are true. In short, the truth of $KB$ logically guarantees the truth of $\alpha$.

### 4. How does theorem proving use satisfiability checking?

**Answer:** Theorem proving uses proof by contradiction (reductio ad absurdum). To check if $KB \models \alpha$, we test whether $KB \land \neg \alpha$ is **unsatisfiable**. If no possible model or truth assignment makes $KB \land \neg \alpha$ true simultaneously, then $KB \models \alpha$ is proven to hold.

### 5. What is one limitation of propositional logic compared to First-Order Logic?

**Answer:** Propositional logic lacks expressiveness regarding objects, relations, and quantifiers ($\forall$, $\exists$). It treats basic propositions as atomic units, meaning general rules (such as "All humans are mortal") cannot be stated concisely and must instead be individually grounded and duplicated for every specific object in the domain (e.g., `Human_Socrates -> Mortal_Socrates`).

## Reflection

### Challenges Encountered

Representing general domain rules (like universal quantifiers in First-Order Logic) using propositional logic required manually or programmatically generating ground propositions for every individual entity, making scalability a major concern as the domain size grows. Translating logical expressions into SymPy and Python structures required careful mapping of operators like `Implies` and `Not` while properly managing variable instances to avoid evaluation or syntax errors during entailment checks. Additionally, conceptualizing why an implication $P \implies Q$ evaluates to true when $P$ is false was initially unintuitive when translating real-world statements into formal logic structures.

### What I Learned

Through this lab, I gained practical experience implementing core logical inference rules like Modus Ponens and Modus Tollens to validate how a knowledge base logically entails specific conclusions. I deepened my understanding of the trade-offs between expressive power and computational simplicity, recognizing that while propositional logic simplifies automated reasoning, First-Order Logic is far more concise for expressing general statements over sets of objects. Furthermore, I learned how automated reasoning systems check entailment ($KB \models \alpha$) by testing the unsatisfiability of $KB \land \neg \alpha$ through proof by contradiction.